# Bedienung

1. Links die Prämienwoche wählen
2. "Run all"

In [ ]:
  import micropip                                                                                                                                                                                                                                                                                                            

   await micropip.install("ipython==8.31.0")
 


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
pd.set_option('display.max_columns', None)

# Header Woche 

In [ ]:
def headerWoche(start_date_dt):
    # Calculate the end date as start date + 7 days
    end_date_dt = start_date_dt + timedelta(days=6)

    # Generate a range of dates
    date_range = pd.date_range(start=start_date_dt, end=end_date_dt)

    # Create a DataFrame with the date range
    df = pd.DataFrame(date_range, columns=['Date'])
    
    df['Wochentag'] = [
        'So',
        'Mo',
        'Di',
        'Mi',
        'Do',
        'Fr',
        'Sa',
    ]
    
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df = df.set_index('Date')
    return df

In [ ]:
def format_column(my_df, col, form):
    my_df[col] = my_df[col].apply(form.format)

In [ ]:
def index_to_date_header(index):
    headers = ['']
    for v in index.values:
        try:
            headers.append(datetime.strptime(v, '%Y-%m-%d %H:%M:%S').strftime('%d.%m.'))
        except:
            headers.append(v)
    #print(headers)
    header = [
        headers, 
        ["","So","Mo","Di","Mi","Do","Fr","Sa","Gesamt"]
             ]
    #print(header)
    return header

In [ ]:
# Function to format numbers in German locale
def format_german_locale(x):
    if str(x) == 'nan':
        return ''
    if isinstance(x, (int, float)):
        return "{:,.2f}".format(x).replace(",", "X").replace(".", ",").replace("X", ".")
    return x.replace(",", "X").replace(".", ",").replace("X", ".")

# Apply the formatting function to the entire DataFrame


# Umsatz

In [ ]:
import numpy as np

def weighted_average(df, r,col1, col2):
    rows = df.loc[r.index]
    #print("==============")
    #print(df.loc[r.index][[col1,col2]])
    sum_top = sum(df.loc[r.index][col1])
    sum_weights = sum(df.loc[r.index][col2])
    #print(sum_top)
    #print(sum_weights)
    if sum_weights == 0:
        return 0
    else:
        weighted_average = sum_top /sum_weights
        #print(weighted_average)    
        return weighted_average

In [ ]:
def umsatz(df):

    umsatz = df.fillna(0).pivot_table(
        index=["Date"], 
        values=[
            'Ist_Umsatz',
            'Anzahl_Kdn_',
            'Sollleistung_EUR_Std',
            'Pramie_rechnerisch_Std_2',
            'PP_StdP',
            'Soll_Umsatz_brutto_2',
            'Minus_Plusumsatz_2',
            'Umsatz_pro_Kunde_Kde_2',
            'Umsatz_pro_PP_Stunde_2',
            'Pramie_Auszahlung_Std_2', 
            'Pramie_Gesamt_3',
            'Pramie_Gesamt_Tag', 
        ], 
        aggfunc={
            'Ist_Umsatz':"sum",
            'Anzahl_Kdn_':"sum",
            'Sollleistung_EUR_Std':"mean",
            'Pramie_rechnerisch_Std_2':"mean",
            'PP_StdP':"sum",
            'Soll_Umsatz_brutto_2':"sum",
            'Minus_Plusumsatz_2':"sum",
            'Umsatz_pro_Kunde_Kde_2': lambda rows: weighted_average(df, rows, 'Ist_Umsatz','Anzahl_Kdn_'),
            'Umsatz_pro_PP_Stunde_2':lambda rows: weighted_average(df, rows, 'Ist_Umsatz','PP_StdP'),
            'Pramie_Auszahlung_Std_2':"mean", 
            'Pramie_Gesamt_3':"mean",
            'Pramie_Gesamt_Tag':"sum", 

        },
        margins=True
    )
    
    format_column(umsatz,'Anzahl_Kdn_','{:,}')
    format_column(umsatz,'Ist_Umsatz','{:,.2f}')
  #  format_column(umsatz,'Minus_Plusumsatz_2','{:,.0f}')
    format_column(umsatz,'PP_StdP','{:,.0f}')
    format_column(umsatz,'Pramie_Auszahlung_Std_2','{:,.2f}')
    format_column(umsatz,'Pramie_Gesamt_3','{:,.2f}')
    format_column(umsatz,'Pramie_Gesamt_Tag','{:,.2f}')
    format_column(umsatz,'Pramie_rechnerisch_Std_2','{:,.2f}')
    format_column(umsatz,'Soll_Umsatz_brutto_2','{:,.2f}')
    format_column(umsatz,'Sollleistung_EUR_Std','{:,.2f}')
    format_column(umsatz,'Umsatz_pro_Kunde_Kde_2','{:,.2f}')
    format_column(umsatz,'Umsatz_pro_PP_Stunde_2','{:,.2f}')

    umsatz = umsatz[[        
        'Ist_Umsatz',
        'Sollleistung_EUR_Std',
        'Soll_Umsatz_brutto_2',
        'Minus_Plusumsatz_2',
        'Anzahl_Kdn_',
        'Umsatz_pro_Kunde_Kde_2',
        'PP_StdP',
        'Umsatz_pro_PP_Stunde_2',
        'Pramie_Gesamt_Tag', 
        'Pramie_Gesamt_3',
        'Pramie_rechnerisch_Std_2',
        'Pramie_Auszahlung_Std_2', 
    ]]
    
    umsatz.columns = [
        'Ist-Umsatz brutto [€]',
        '(Sollleistung [€/Std.])',
        'Soll-Umsatz brutto [€]',
        'Minus-/Plusumsatz [€]',
        'Kundenanzahl [St.]',
        'Umsatz pro Kunde [€/Kde.]',
        'PP-Stunden [Std.]',
        'Umsatz pro PP-Stunde [€]',
        'Prämie Gesamt Tag [€]',
        'Prämie Gesamt Woche [€]',
        'Prämie rechnerisch [€/Std.]',
        'Prämie Auszahlung [€/Std.]',
        ]
    
    umsatz.index = umsatz.index.astype(str)
    # don't show the values for Prämien... except for the total
    
    umsatz = headers.merge(umsatz, how='outer', left_index=True, right_index=True, suffixes=('',''))
    umsatz = umsatz.drop(columns=['Wochentag', 'Prämie Gesamt Woche [€]'])
    umsatz = umsatz.applymap(format_german_locale)
    umsatz.iloc[0:7,8:11] = '' 
    return umsatz

In [ ]:
dfu = pd.DataFrame(await grist.fetch_table("Umsatz"))
dfu['Date'] = pd.to_datetime(dfu['Datum'], unit='s')

In [ ]:
dfu.head()

# Zeit

In [ ]:
dfz = pd.DataFrame(await grist.fetch_table("Zeit2"))
dfz['Date'] = pd.to_datetime(dfz['Datum'])

In [ ]:
#dfz.head()

In [ ]:
def zeiten(df):
    zeiten = df.pivot_table(
        index=["Date"],
        columns=["gristHelper_Display"],
        values=["geleistete_Stunden"], 
        aggfunc={"geleistete_Stunden":"sum"},
        margins=True
    ).fillna("")
    
    zeiten.columns = zeiten.columns.droplevel()
    zeiten.index = zeiten.index.astype(str)
    
    
    zeiten = headers.merge(zeiten, how='outer', left_index=True, right_index=True, suffixes=('',''))
    zeiten = zeiten.drop(columns=['Wochentag'])
    zeiten = zeiten.applymap(format_german_locale)

    return zeiten

In [ ]:
from IPython.display import display, Javascript, Markdown as md, HTML
header = """
    <div id="_my_special_div"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/html2canvas/1.4.1/html2canvas.min.js" integrity="sha512-BNaRQnYJYiPSqHHDb58B0yaPfCu+Wgds8Gp/gU33kqBtgNS4tSPHuGibyoeqMV/TJlSKda6FXzoEyYGjTe+vXA==" crossorigin="anonymous" referrerpolicy="no-referrer" defer></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/dompurify/3.1.6/purify.min.js" integrity="sha512-jB0TkTBeQC9ZSkBqDhdmfTv1qdfbWpGE72yJ/01Srq6hEzZIz2xkz1e57p9ai7IeHMwEG7HpzG6NdptChif5Pg==" crossorigin="anonymous" referrerpolicy="no-referrer" defer></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js" defer></script>
    <script id="myScriptId" src="https://cdnjs.cloudflare.com/ajax/libs/jspdf-autotable/3.8.3/jspdf.plugin.autotable.min.js" integrity="sha512-CNhbAXGrvqmSMpgydAOp7SFP50hdDaIMir76cuptjMdG0V4Hq7h+JzpSudrD2mRC38tawkA8aQSU8dQ0H6hzdQ==" crossorigin="anonymous" referrerpolicy="no-referrer" defer></script>
    <script defer>

    // Function to wait for thatObject to become available
    function waitForJSPDF() {{
        return new Promise((resolve, reject) => {{
            const intervalId = setInterval(() => {{
                if (window.jspdf) {{
                    clearInterval(intervalId);
                    resolve(window.jspdf);
                }}
            }}, 100); // Check every 100 milliseconds for availability of thatObject
        }});
    }}
    
    // Function to wait for thatObject to become available
    function waitForAutoTable(md) {{
        return new Promise((resolve, reject) => {{
            console.log("auto promise");
            const doc = new jspdf.jsPDF('p','pt','a4')
            const intervalId = setInterval(() => {{
                console.log("waiting auto ..");
                if (doc.autoTable) {{
                    clearInterval(intervalId);
                    resolve(md.autoTable);
                }}
            }}, 100); // Check every 100 milliseconds for availability of thatObject
            setTimeout(() => {{
                clearInterval(intervalId);
                reject("autoTable not available after timeout");
            }}, 5000);

        }});
    }}
    
    var myDocs = [
""".format()
footer = """

    ]; 
    
    function my_autoTable(mydoc, cols, rows) {{
        console.log(cols);
        mydoc.autoTable({{
            head: cols,
            body: rows,
            columnStyles: {{
                0: {{cellWidth: 120}},
                1: {{cellWidth: 50, halign: 'right'}},
                2: {{cellWidth: 50, halign: 'right'}},
                3: {{cellWidth: 50, halign: 'right'}},
                4: {{cellWidth: 50, halign: 'right'}},
                5: {{cellWidth: 50, halign: 'right'}},
                6: {{cellWidth: 50, halign: 'right'}},
                7: {{cellWidth: 50, halign: 'right'}},
                8: {{cellWidth: 60, halign: 'right', fontStyle: 'bold'}},
            }},
            styles: {{
                fontSize: 9
            }},
        }})
    }}

   function Document(filiale, cols1, rows1, cols2, rows2) {{
        this.filiale = filiale;
        this.cols1 = cols1;
        this.rows1 = rows1;
        this.cols2 = cols2;
        this.rows2 = rows2;
    }}
    
    function reloadScript(scriptId) {{
      const oldScript = document.getElementById(scriptId);
      console.log(oldScript);
      if (oldScript) {{
        const newScript = document.createElement('script');
        newScript.src = oldScript.src;
        newScript.id = oldScript.id;
        oldScript.parentNode.replaceChild(newScript, oldScript);
      }}
    }}
    
    
    function Wait() {{

        // Call the function with the ID of your script tag
        reloadScript('myScriptId');

    
        //window.jsPDF = window.jspdf.jsPDF;
        waitForJSPDF().then(() => {{
            console.log("waited for JSPDF");
            // Use thatObject functionalities here
            if (typeof window.jsPDF === 'undefined') {{
                // window.jsPDF = window.jspdf.jsPDF;
            }}
            waitForAutoTable(new jspdf.jsPDF('p', 'pt','a4')).then(() => {{
                console.log("waited for Autotable");
                generateDocs();
            }});
        }});

    }}

        
    function generateDocs() {{
        console.log("Generation started");
        myDocs.forEach(d => {{
            // Only pt supported (not mm or in)
            var doc = new jspdf.jsPDF('p', 'pt','a4');

            doc.text(20, 20, "Wochenübersicht: " + d.filiale)
            my_autoTable(doc, d.cols1, d.rows1);
            my_autoTable(doc, d.cols2, d.rows2);
            doc.save(d.filiale + '.pdf');
        }}); 
    }}

  </script>
  <button id="alertButton" onclick="Wait()">Berichte runterladen!</button>
""".format()

def js_convert_str_html(title, u, z):
    js_convert = """
        new Document({0},{1},{2},{3},{4}),
    """.format(
       title,
        index_to_date_header(u.index),
        u.T.reset_index().to_json(orient='values'),
        index_to_date_header(z.index),
        z.T.reset_index().to_json(orient='values')
    )
    
    return js_convert  



In [ ]:
filialen = pd.DataFrame(await grist.fetch_table("KST"))

In [ ]:
debug = dfu[(dfu['Filiale'] == 5) & (dfu['PW'] == 38)]

In [ ]:
#debug

In [ ]:
u = umsatz(dfu[(dfu['Filiale'] == 18) & (dfu['PW'] == 42)])
u

In [ ]:
Woche = await grist.fetch_selected_record()
headers = headerWoche(Woche['PW'])
PW = Woche["id"]
print('Ausgewählte Woche: {0}'.format(Woche['PW']))

body = header

for i,f in filialen.iterrows():
    #Filiale Leutkirch wird übersprungen da geschlossen seit 28.03.2025, falls man es doch will, dann die Zahl ändern auf eine Zahl die nicht bei den Filialen verwendet wird, bsp 25)
    if f['id']!=18:    
        print("Auswertung für Filiale {0} {1} {2}".format(f['id'], PW, f['Bezeichnung']))
        try:
            u = umsatz(dfu[(dfu['Filiale'] == f['id']) & (dfu['PW'] == PW)])
            z = zeiten(dfz[(dfz['KST'] == f['id']) & (dfz['Pramienwoche'] == PW)])

            body = body + js_convert_str_html( "'{0} {1}'".format(Woche['Pramienwochen_Index'],f['Bezeichnung']),u,z)
        except Exception as e:
            print("Fehler mit Filiale {0} {1} bitte Tabellen 'Zeit2' und 'Umsatz' checken ob Fehler in den hinteren Spalten sind? Z.B. fehlendende PP Werte für MA".format(str(f['id']),f['Bezeichnung']))
            print(e.msg)


body = body + footer
display(HTML(body))
#print(body)